# 01. Toolkit Playground

**Purpose:** Welcome notebook to get hands-on with the toolkit. You will run a *single* API call and see how decoding parameters affect output quality and consistency.


## Step 1: Install Dependencies

In [ ]:
%pip install -q -r ../requirements.txt

## Step 2: Load Shared Utilities

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Configure the API Client

In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ.get('WORKSHOP_EMAIL', '')

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')
print('STUDENT_EMAIL =', STUDENT_EMAIL)


## Step 4: Single-Address Prompt + Parameter Playground (edit **one cell only**)

**Goal:** Try different prompts and observe how decoding parameters affect the model’s behavior.

**What to edit in the next cell:**
- `PROMPT_TEMPLATE` (your instructions)
- `INPUT_ADDRESS` (optional)
- `TEMPERATURE`, `TOP_P`, `TOP_K`

**Suggested experiments (do these in order):**
1. Run once with **`TEMPERATURE = 0.0`** (most consistent).
2. Increase temperature (e.g., **0.2 → 0.7**) and re-run. Watch for more variability and occasional formatting drift.
3. Keep temperature fixed and vary **`TOP_P`** (e.g., 0.8 vs 1.0) to see how nucleus sampling changes outputs.
4. Keep temperature/top_p fixed and vary **`TOP_K`** (e.g., 20 vs 80) to see how restricting candidate tokens affects stability.

**Note:** For extraction tasks, lower randomness is usually better—but you should *observe* the trade-off yourself.


In [ ]:
import json

# -----------------------------
# INPUT (edit if you want)
# -----------------------------
INPUT_ADDRESS = "332 Menzies Street, Victoria, BC V8V 2G9, Canada"

# -----------------------------
# PROMPT (edit here)
# -----------------------------
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

# -----------------------------
# MODEL + DECODING PARAMS (edit these)
# -----------------------------
MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')

TEMPERATURE = 0.0   # try: 0.0, 0.2, 0.7
TOP_P = 0.95        # try: 0.8, 0.95, 1.0
TOP_K = 40          # try: 20, 40, 80

# Stage should be valid. For playground, use baseline.
STAGE_NAME = 'baseline'

# -----------------------------
# EXECUTE (single address)
# -----------------------------
result = process_single_address(
    INPUT_ADDRESS,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    custom_prompt_template=PROMPT_TEMPLATE,
)

# -----------------------------
# DISPLAY
# -----------------------------
print("Raw result:")
print(result)

# Pretty-print if result is JSON-serializable
try:
    print("\nPretty JSON:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
except TypeError:
    pass
